In [1]:
# 学習
import torch
import torch.optim as optim
from yolov3.models.yolo import Model
import yaml
from tqdm import tqdm
from src.domain.loss import CustomLoss
from src.domain.dataloader import CustomDataset, custom_collate_fn
from torch.utils.data import DataLoader

# モデルのロード
config_path = 'yolov3/models/yolov5s.yaml'
model_path = 'models/pre_trained/yolov5s.pt'
model = Model(config_path)
model.load_state_dict(torch.load(model_path)['model'].state_dict())  # yolov5s.ptは、学習済みの重みファイル
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
model.to(device)

hyp_path = "yolov3/data/hyps/hyp.scratch-low.yaml"
with open(hyp_path, errors="ignore") as f:
    hyp = yaml.safe_load(f)

model.hyp = hyp

# カスタム損失関数
criterion = CustomLoss(model)

# オプティマイザ
optimizer = optim.Adam(model.parameters(), lr=0.002)
# scaler = torch.cuda.amp.GradScaler(enabled=True) # 高速化ライブラリ必要であれば利用したい

# データローダ
img_dir = './data/coco128/images/train2017'
annotation_dir = './data/coco128/labels/train2017'
train_dataset = CustomDataset(
    img_dir=img_dir,
    annotation_dir=annotation_dir,
)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=custom_collate_fn)

# tqdmの表示フォーマット
TQDM_BAR_FORMAT = '{l_bar}{bar:10}{r_bar}'

# トレーニングループ
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    pbar = tqdm(train_loader, total=len(train_loader), bar_format=TQDM_BAR_FORMAT)
    for images, targets in pbar:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss, loss_items = criterion(outputs, targets)
        # scaler.scale(loss).backward()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        # a
        pbar.set_description(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader)}')
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {running_loss / len(train_loader)}')
    

# トレーニング済みモデルの保存
torch.save(model.state_dict(), 'models/fine_tuned/yolov5s_finetuned.pth')
print("model save!")


/home/docker/.cache/pypoetry/virtualenvs/repo-uZbbGesQ-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

                 from  n    params  module                                  arguments                     
  0                -1  1      3520  models.common.Conv                      [3, 32, 6, 2, 2]              
  1                -1  1     18560  models.common.Conv                      [32, 64, 3, 2]                
  2                -1  1     18816  models.common.C3                        [64, 64, 1]                   
  3                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  4                -1  2    115712  models.common.C3                        [128, 128, 2]                 
  5                -1  1    295424  models.common.Conv       

cuda


Epoch [1/20], Loss: 1.0811878126114607: 100%|██████████| 32/32 [00:08<00:00,  3.70it/s] 


Epoch [1/20], Loss: 1.0811878126114607


Epoch [2/20], Loss: 0.9942306689918041: 100%|██████████| 32/32 [00:04<00:00,  7.30it/s] 


...

Epoch [18/20], Loss: 0.9319030921906233


Epoch [19/20], Loss: 0.9220664072781801: 100%|██████████| 32/32 [00:04<00:00,  7.03it/s] 


Epoch [19/20], Loss: 0.9220664072781801


Epoch [20/20], Loss: 0.9231257326900959: 100%|██████████| 32/32 [00:04<00:00,  7.13it/s] 

Epoch [20/20], Loss: 0.9231257326900959
model save!


In [7]:
print(len(outputs))
for i in range(3):
    print(outputs[i].shape)

index = torch.tensor([0,0])

outputs[0][index,index,index,index].split((2, 2, 1, 80), 1)


3
torch.Size([2, 3, 72, 72, 85])
torch.Size([2, 3, 36, 36, 85])
torch.Size([2, 3, 18, 18, 85])


(tensor([[0.47582, 3.97380],
         [0.47582, 3.97380]], device='cuda:0', grad_fn=<SplitWithSizesBackward0>),
 tensor([[2.07238, 0.12014],
         [2.07238, 0.12014]], device='cuda:0', grad_fn=<SplitWithSizesBackward0>),
 tensor([[-15.14550],
         [-15.14550]], device='cuda:0', grad_fn=<SplitWithSizesBackward0>),
 tensor([[ -1.31783,  -6.09178,  -6.39082, -33.91017, -35.85278, -10.96098, -37.50455,  -9.68083, -31.98620, -13.59475, -35.73943, -35.26254, -32.51019, -16.48398,  -6.61868, -29.27445, -19.88538, -31.72448, -28.89687, -25.03902, -34.21565, -31.22474, -23.31136, -29.83593, -27.03916, -32.64078, -17.39379, -25.41164,
          -31.68935, -13.60621, -25.51579, -30.12466,  -6.68450, -14.04212, -31.91874,  -7.87903, -20.62366, -30.09458, -11.59782,  -3.50834, -27.05721,  -4.38701, -19.80958, -20.17062,  -2.26467, -25.74683, -29.21626, -25.18502, -28.79195, -24.21514, -20.99507, -11.56283, -30.21341, -27.85118, -25.75468, -21.18832,
           -6.89479, -32.43427, -23.45564,

# 推論用のデータ読み込み機能

In [1]:
import cv2
import numpy as np
import os
from pathlib import Path
import glob

IMG_FORMATS = "bmp", "dng", "jpeg", "jpg", "mpo", "png", "tif", "tiff", "webp", "pfm"  # include image suffixes

def letterbox(im, new_shape=(640, 640), color=(114, 114, 114), auto=True, scaleFill=False, scaleup=True, stride=32):
    """Resizes and pads an image to a new shape with optional scaling, filling, and stride-multiple constraints."""
    shape = im.shape[:2]  # current shape [height, width]
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    # Scale ratio (new / old)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    if not scaleup:  # only scale down, do not scale up (for better val mAP)
        r = min(r, 1.0)

    # Compute padding
    ratio = r, r  # width, height ratios
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]  # wh padding
    if auto:  # minimum rectangle
        dw, dh = np.mod(dw, stride), np.mod(dh, stride)  # wh padding
    elif scaleFill:  # stretch
        dw, dh = 0.0, 0.0
        new_unpad = (new_shape[1], new_shape[0])
        ratio = new_shape[1] / shape[1], new_shape[0] / shape[0]  # width, height ratios

    dw /= 2  # divide padding into 2 sides
    dh /= 2

    if shape[::-1] != new_unpad:  # resize
        im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    im = cv2.copyMakeBorder(im, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)  # add border
    return im, ratio, (dw, dh)

class LoadImages:
    # YOLOv3 image/video dataloader, i.e. `python detect.py --source image.jpg/vid.mp4`
    def __init__(self, path, img_size=640, stride=32, auto=True, transforms=None):
        """Initializes the data loader for YOLOv3, supporting image, video, directory, and '*.txt' path lists with
        customizable image sizing.
        """
        if isinstance(path, str) and Path(path).suffix == ".txt":  # *.txt file with img/vid/dir on each line
            path = Path(path).read_text().rsplit()
        files = []
        for p in sorted(path) if isinstance(path, (list, tuple)) else [path]:
            p = str(Path(p).resolve())
            if "*" in p:
                files.extend(sorted(glob.glob(p, recursive=True)))  # glob
            elif os.path.isdir(p):
                files.extend(sorted(glob.glob(os.path.join(p, "*.*"))))  # dir
            elif os.path.isfile(p):
                files.append(p)  # files
            else:
                raise FileNotFoundError(f"{p} does not exist")

        images = [x for x in files if x.split(".")[-1].lower() in IMG_FORMATS]
        ni = len(images)

        self.img_size = img_size
        self.stride = stride
        self.files = images
        self.nf = ni  # number of files
        self.mode = "image"
        self.auto = auto
        self.transforms = transforms  # optional
        assert self.nf > 0, (
            f"No images or videos found in {p}. "
            f"Supported formats are:\nimages: {IMG_FORMATS}"
        )

    def __iter__(self):
        """Initializes the iterator by resetting count to zero and returning the iterator instance itself."""
        self.count = 0
        return self

    def __next__(self):
        """Advances to the next file in the dataset, raising StopIteration when all files are processed."""
        if self.count == self.nf:
            raise StopIteration
        path = self.files[self.count]

        # Read image
        self.count += 1
        im0 = cv2.imread(path)  # BGR
        assert im0 is not None, f"Image Not Found {path}"
        s = f"image {self.count}/{self.nf} {path}: "

        if self.transforms:
            im = self.transforms(im0)  # transforms
        else:
            im = letterbox(im0, self.img_size, stride=self.stride, auto=self.auto)[0]  # padded resize
            im = im.transpose((2, 0, 1))[::-1]  # HWC to CHW, BGR to RGB
            im = np.ascontiguousarray(im)  # contiguous

        return path, im, im0, s

    def __len__(self):
        """Returns the number of files in the dataset."""
        return self.nf  # number of files

# 推論

In [53]:
# 推論
import torch
import torch.optim as optim
from yolov3.models.yolo import Model
import yaml
from tqdm import tqdm
from src.domain.loss import CustomLoss
from src.domain.dataloader import CustomDataset, custom_collate_fn
from torch.utils.data import DataLoader

from torchvision import transforms
from PIL import Image


from yolov3.utils.general import non_max_suppression
from ultralytics.utils.plotting import Annotator, colors, save_one_box
from yolov3.utils.general import (
    LOGGER,
    Profile,
    check_file,
    check_img_size,
    check_imshow,
    check_requirements,
    colorstr,
    cv2,
    increment_path,
    non_max_suppression,
    print_args,
    scale_boxes,
    strip_optimizer,
    xyxy2xywh,
)

# モデルのロード
config_path = 'yolov3/models/yolov5s.yaml'
model = Model(config_path)

model_path = 'models/fine_tuned/yolov5s_finetuned.pth'
model.load_state_dict(torch.load(model_path))
# model_path = 'models/pre_trained/yolov5s.pt'
# model.load_state_dict(torch.load(model_path)['model'].state_dict())  # yolov5s.ptは、学習済みの重みファイル


                 from  n    params  module                                  arguments                     
  0                -1  1      3520  models.common.Conv                      [3, 32, 6, 2, 2]              
  1                -1  1     18560  models.common.Conv                      [32, 64, 3, 2]                
  2                -1  1     18816  models.common.C3                        [64, 64, 1]                   
  3                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  4                -1  2    115712  models.common.C3                        [128, 128, 2]                 
  5                -1  1    295424  models.common.Conv                      [128, 256, 3, 2]              
  6                -1  3    625152  models.common.C3                        [256, 256, 3]                 
  7                -1  1   1180672  models.common.Conv                      [256, 512, 3, 2]              
  8                -1  1   1182720  

YOLOv3s summary: 214 layers, 7235389 parameters, 7235389 gradients, 16.6 GFLOPs



<All keys matched successfully>

In [54]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 推論モード
model.eval()

image_path = "data/images/bus.jpg"
im0 = cv2.imread(image_path)
image = Image.open(image_path).convert("RGB")
transform = transforms.Compose([
            transforms.Resize((576, 576)),
            transforms.ToTensor(),
        ])
image = transform(image).unsqueeze(0)
# 推論
with torch.no_grad():  # メモリ効率を向上させるためにtorch.no_gradを使用
    input = image.to(device)
    pred = model(input)

# 推論結果を表示または処理するコードを追加
print(pred)

(tensor([[[1.11342e+01, 1.18127e+01, 3.99999e+01,  ..., 0.00000e+00, 2.16664e-21, 7.09265e-10],
         [1.96991e+01, 1.18700e+01, 4.00000e+01,  ..., 0.00000e+00, 3.25763e-25, 8.95283e-10],
         [2.77283e+01, 1.18646e+01, 4.00000e+01,  ..., 0.00000e+00, 5.25493e-25, 1.20926e-09],
         ...,
         [5.02983e+02, 5.44723e+02, 7.80995e+01,  ..., 7.72586e-03, 1.47109e-06, 3.03525e-03],
         [5.30296e+02, 5.47346e+02, 6.05090e+01,  ..., 9.29869e-03, 1.20626e-06, 2.28170e-03],
         [5.68309e+02, 5.40684e+02, 6.33361e+01,  ..., 9.20519e-03, 8.49649e-07, 3.64958e-03]]], device='cuda:0'), [tensor([[[[[ 2.86103e+00,  4.43607e+00,  1.38329e+01,  ..., -1.27839e+02, -4.75811e+01, -2.10668e+01],
           [ 3.95466e+00,  4.80455e+00,  1.87840e+01,  ..., -1.70330e+02, -5.63836e+01, -2.08339e+01],
           [ 4.05870e+00,  4.76347e+00,  1.86732e+01,  ..., -1.68764e+02, -5.59055e+01, -2.05333e+01],
           ...,
           [ 1.64145e+00,  2.28375e+00,  5.45528e+00,  ..., -5.52921e

In [55]:
print(isinstance(pred, (list, tuple)))
print(len(pred))
print(pred[0].shape)
print(len(pred[1]))
print(pred[1][0].shape)
print(pred[1][1].shape)
print(pred[1][2].shape)
print(pred[0].device)

pred[0][..., 4].max()

True
2
torch.Size([1, 20412, 85])
3
torch.Size([1, 3, 72, 72, 85])
torch.Size([1, 3, 36, 36, 85])
torch.Size([1, 3, 18, 18, 85])
cuda:0


tensor(0.09582, device='cuda:0')

In [42]:
output = non_max_suppression(pred, conf_thres=0.01)
print(output[0].shape)
print(output[0])
print(output[0].cpu().numpy())
print(model.names)

torch.Size([78, 6])
tensor([[-7.16583e-01,  4.27715e+01,  5.70497e+02,  5.66451e+02,  3.12673e-02,  0.00000e+00],
        [ 3.16858e+00,  4.91066e+01,  5.72644e+02,  5.49727e+02,  2.23191e-02,  6.00000e+01],
        [ 9.66436e+00,  3.12510e+02,  1.11628e+02,  5.65457e+02,  2.15691e-02,  0.00000e+00],
        [-1.69450e+00,  2.41029e+02,  1.51049e+02,  5.14825e+02,  2.11174e-02,  0.00000e+00],
        [ 8.40911e+01,  3.17181e+02,  1.71011e+02,  4.98136e+02,  1.89110e-02,  0.00000e+00],
        [ 3.31374e+02,  4.65388e+02,  3.65018e+02,  4.98921e+02,  1.73097e-02,  5.10000e+01],
        [ 2.44364e+02,  1.87807e+02,  3.42511e+02,  5.41424e+02,  1.53675e-02,  0.00000e+00],
        [ 3.17043e+02,  3.73210e+02,  3.61909e+02,  4.27519e+02,  1.51362e-02,  2.00000e+00],
        [ 3.28466e+02,  3.03013e+02,  3.48900e+02,  3.82622e+02,  1.49270e-02,  4.00000e+01],
        [ 1.62895e+02,  2.98771e+02,  2.59423e+02,  4.85127e+02,  1.48207e-02,  0.00000e+00],
        [ 3.40356e+00,  7.58032e+00,  5.

In [23]:
save_crop=False  # save cropped prediction boxes
line_thickness=3  # bounding box thickness (pixels)
save_txt=False  # save results to *.txt
view_img=True  # show results
nosave=False  # do not save images/videos
hide_labels=False  # hide labels
hide_conf=False  # hide confidences

names = model.names
# im0 = im0.to(device)
im = image.to(device)
output_path = "data/output/test/"

print("読み込み完了")
# Process predictions
for i, det in enumerate(output):  # per image
    print("処理開始")
    # det = det.cpu()
    gn = torch.tensor(im0.shape)[[1, 0, 1, 0]]  # normalization gain whwh
    imc = im0.copy() if save_crop else im0  # for save_crop
    annotator = Annotator(im0, line_width=line_thickness, example=str(names))
    if len(det):
        # Rescale boxes from img_size to im0 size
        det[:, :4] = scale_boxes(im.shape[2:], det[:, :4], im0.shape).round()

        # Print results
        for c in det[:, 5].unique():
            n = (det[:, 5] == c).sum()  # detections per class
            # s += f"{n} {names[int(c)]}{'s' * (n > 1)}, "  # add to string

        # Write results
        for *xyxy, conf, cls in reversed(det):
            print("annotator")
            # if save_txt:  # Write to file
            #     xywh = (xyxy2xywh(torch.tensor(xyxy).view(1, 4)) / gn).view(-1).tolist()  # normalized xywh
            #     line = (cls, *xywh, conf) if save_conf else (cls, *xywh)  # label format
            #     with open(f"{txt_path}.txt", "a") as f:
            #         f.write(("%g " * len(line)).rstrip() % line + "\n")

            c = int(cls.cpu())  # integer class
            label = None if hide_labels else (names[c] if hide_conf else f"{names[c]} {conf:.2f}")
            annotator.box_label(xyxy, label, color=colors(c, True))

    result_image = annotator.result()
    # cv2.imshow("result", result_image)
    cv2.imwrite(output_path+"bus.jpg", result_image)


読み込み完了
処理開始
